# Laboratorio 10 — O Pipeline Definitivo
## RAG, QLoRA e Otimizacao de Inferencia na GPU
---
> **Declaracao:** Partes deste laboratorio foram geradas/complementadas com IA, revisadas e validadas por **Andre Lucas Francino Castelo Branco**.
>
> **Ambiente recomendado:** Google Colab com GPU T4 — *Runtime → Change runtime type → T4 GPU*

## Instalacao de Dependencias
> `flash-attn` pode levar **5 a 10 minutos** para compilar. Aguarde a celula terminar antes de prosseguir.

In [ ]:
!pip install -q 'transformers>=4.41.0' 'bitsandbytes>=0.43.1' 'accelerate>=0.30.0'
print('Dependencias instaladas!')
print('Nota: usaremos attn_implementation=sdpa (PyTorch 2.x nativo) — equivalente ao FlashAttention-2.')


---
## Passo 1 — Ingestao Eficiente: Carregando o Modelo com QLoRA 4-bit

Carregamos o modelo `TinyLlama-1.1B` quantizado em **NF4 (NormalFloat4)** via `bitsandbytes`,
reduzindo o footprint de VRAM aproximadamente **4x** em relacao ao Float16 original.
Os calculos internos ainda ocorrem em Float16 para preservar qualidade.

**Metrica coletada:** VRAM ocupada apos a carga.

In [ ]:
import os
import torch
import time
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Reduz fragmentacao de VRAM — deve ser definido antes de qualquer alocacao CUDA
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

assert torch.cuda.is_available(), 'GPU nao detectada! Va em Runtime > Change runtime type > T4 GPU'

device_name = torch.cuda.get_device_name(0)
vram_total_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU detectada : {device_name}')
print(f'VRAM total    : {vram_total_gb:.1f} GB')

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
)

print('\nCarregando tokenizador...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print('Carregando modelo em 4-bit (sem FlashAttention-2)...')
torch.cuda.reset_peak_memory_stats()
torch.cuda.empty_cache()

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
)
model.eval()

vram_modelo_mb = torch.cuda.memory_allocated() / 1024**2

print()
print('[PASSO 1 - METRICA]')
print('=' * 50)
print(f'VRAM apos carga do modelo (4-bit): {vram_modelo_mb:.1f} MB')
print('=' * 50)


---
## Passo 2 — Simulando o RAG Massivo

Geramos uma string de texto medico ficticio que simula os capitulos recuperados pelo banco vetorial.
O tokenizador trunca o texto no tamanho desejado.

- **Passo 3 (sem FA2):** contexto de **4.000 tokens** — limite seguro sem FlashAttention
- **Passo 4 (com FA2):** contexto de **12.000 tokens** — escala completa com otimizacao

In [ ]:
TEXTO_MEDICO_RAG = '''
[CAPITULO 1 - AVALIACAO CARDIOVASCULAR EM PACIENTES DE ALTO RISCO]

Paciente J.M.S., 67 anos, sexo masculino, admitido com dispneia aos medios esforcos,
edema bilateral de membros inferiores e palpitacoes intermitentes ha tres semanas.
Historico: hipertensao arterial sistemica ha 15 anos, diabetes mellitus tipo 2 com HbA1c 9,1%,
dislipidemia mista, sindrome coronariana aguda sem supradesnivelamento do ST ha 18 meses.
Medicacao atual: metformina 850mg 2x/dia, glibenclamida 5mg/dia, atorvastatina 40mg/noite,
losartana 100mg/dia, AAS 100mg/dia, clopidogrel 75mg/dia, bisoprolol 5mg/dia, furosemida 40mg/dia.
ECG: fibrilacao atrial com resposta ventricular controlada em torno de 75bpm, sem bloqueios novos.
Ecocardiograma: fracao de ejecao 42%, dilatacao moderada de atrio esquerdo, regurgitacao mitral leve.
Laboratorial: glicemia 186mg/dL, creatinina 1,4mg/dL com TFG estimada de 52mL/min/1,73m2,
BNP 820pg/mL, troponina negativa, PCR 4,2mg/L, sodio 138mEq/L, potassio 4,1mEq/L.
Conduta: ajuste de diuretico para furosemida 80mg/dia, inicio de sacubitril/valsartana 24/26mg 2x/dia,
encaminhamento para eletrofisiologia para ablacao de FA caso refratario ao tratamento medicamentoso,
otimizacao glicemica com insulina basal NPH 10UI ao deitar e monitoracao glicemica 4x/dia.

[CAPITULO 2 - MANEJO DE DIABETES EM CONTEXTO HOSPITALAR]

O controle glicemico em ambiente hospitalar requer atencao especial para evitar hipo e hiperglicemia.
Protocolos baseados em evidencias recomendam manutencao de glicemia entre 140 e 180mg/dL em pacientes criticos.
Para pacientes nao criticos fora de UTI, alvo entre 100 e 180mg/dL antes das refeicoes e ao deitar.
O protocolo basal-bolus e preferivel ao regime de escala movel isolado por proporcionar controle mais estavel.
Pacientes com doenca renal cronica devem ter a dose de metformina ajustada conforme TFG:
manter com TFG entre 45 e 60, reduzir com TFG entre 30 e 45, suspender se TFG abaixo de 30.
Monitoracao de hipoglicemia e mandatoria especialmente durante o jejum pre-procedimento.
Os inibidores de SGLT2 apresentam evidencia robusta de cardioproteção e nefroproteção:
empagliflozina reduziu MACE em 14% e hospitalizacao por insuficiencia cardiaca em 35% no EMPA-REG OUTCOME.
Canagliflozina reduziu progressao de doenca renal em 34% no estudo CREDENCE.
Os agonistas de GLP-1 como semaglutida demonstraram reducao de 26% em MACE no SUSTAIN-6.

[CAPITULO 3 - INSUFICIENCIA RENAL CRONICA E COMORBIDADES ASSOCIADAS]

A DRC estagio G3a com TFG entre 45 e 59 mL/min esta associada a risco cardiovascular aumentado.
Proteinuria acima de 300mg/g de creatinina indica progressao acelerada e necessita avaliacao nefrologica.
Os IECA e BRA sao primeira linha no tratamento da DRC diabetica e ou com proteinuria significativa.
Anemia da DRC: alvo de hemoglobina entre 10 e 11,5 g/dL; avaliar estoques de ferro antes de iniciar EPO.
Doenca mineral ossea renal: monitorar PTH, fosforo, vitamina D e calcemia trimestralmente a partir de G3b.
Contraindicados na DRC avancada: AINEs por serem nefrotoxicos, contraste iodado sem hidratacao adequada,
aminoglicosideos por serem nefrotoxicos e ototoxicos, gadolinio em TFG abaixo de 30.

[CAPITULO 4 - ANTICOAGULACAO EM FIBRILACAO ATRIAL NAO VALVAR]

A escala CHA2DS2-VASc deve ser calculada em todos os pacientes com FA diagnosticada.
Anticoagulacao indicada para CHA2DS2-VASc maior ou igual a 2 em homens e maior ou igual a 3 em mulheres.
Os anticoagulantes orais de acao direta sao preferidos aos antagonistas de vitamina K.
Rivaroxabana 20mg uma vez ao dia com refeicao; reduzir para 15mg se TFG entre 15 e 50.
Apixabana 5mg duas vezes ao dia; reduzir para 2,5mg se dois ou mais dos seguintes:
idade acima de 80 anos, peso abaixo ou igual a 60kg, creatinina acima ou igual a 1,5mg/dL.
Dabigatrana 150mg duas vezes ao dia; contraindicada se TFG abaixo de 30mL/min.

[CAPITULO 5 - DISLIPIDEMIA E RISCO CARDIOVASCULAR RESIDUAL]

O alvo de LDL para pacientes de muito alto risco cardiovascular e abaixo de 50mg/dL.
Estatinas de alta intensidade sao a primeira escolha: atorvastatina 40 a 80mg ou rosuvastatina 20 a 40mg.
Se o LDL permanecer acima do alvo com dose maxima tolerada de estatina, adicionar ezetimiba 10mg/dia.
Se ainda acima do alvo considerar inibidores de PCSK9 como alirocumabe 75mg ou evolocumabe 140mg.
Monitorar transaminases e CPK periodicamente: miopatia e rara mas potencialmente grave.
''' * 25

TOKENS_BASELINE = 4000
TOKENS_FA2 = 5000  # Reducao de 12k para 5k: seguro na T4 com SDPA memory-efficient

def criar_input(n_tokens):
    enc = tokenizer(
        TEXTO_MEDICO_RAG,
        return_tensors='pt',
        truncation=True,
        max_length=n_tokens,
        padding=False,
    ).to('cuda')
    print(f'  Tokens no input: {enc["input_ids"].shape[1]:,}')
    return enc

print('Criando contexto para Passo 3 (sem FA2):')
inputs_baseline = criar_input(TOKENS_BASELINE)
print('Contexto baseline pronto!')


---
## Passo 3 — O Gargalo: Geracao SEM KV Cache

Com `use_cache=False`, o modelo **recalcula todos os vetores Q, K e V** para os tokens anteriores
a cada novo token gerado. Isso gera complexidade efetiva **O(n²)** no decoding:

- Cada passo de geracao custa um forward pass completo sobre todos os tokens do contexto
- 100 tokens gerados = 100 forward passes completos
- Resultado: geracao lenta e pico de VRAM elevado

> Aguarde 1–3 minutos para esta celula concluir.

In [ ]:
print('[PASSO 3] Geracao SEM KV Cache')
print('Contexto: ' + str(TOKENS_BASELINE) + ' tokens | Novos tokens: 100')
print('Aguarde — pode demorar 1-3 minutos...')
print()

model.config.use_cache = False
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t0 = time.perf_counter()

with torch.no_grad():
    out_sem_cache = model.generate(
        **inputs_baseline,
        max_new_tokens=100,
        do_sample=False,
        use_cache=False,
        pad_token_id=tokenizer.eos_token_id,
    )

t_sem_cache = time.perf_counter() - t0
vram_peak_sem = torch.cuda.max_memory_allocated() / 1024**2
n_novos_sem = out_sem_cache.shape[1] - inputs_baseline['input_ids'].shape[1]

print('[PASSO 3 - METRICA]')
print('=' * 52)
print(f'Contexto              : {TOKENS_BASELINE:,} tokens')
print(f'Novos tokens gerados  : {n_novos_sem}')
print(f'Tempo total           : {t_sem_cache:.2f} s')
print(f'Tokens por segundo    : {n_novos_sem / t_sem_cache:.2f}')
print(f'Pico de VRAM          : {vram_peak_sem:.1f} MB')
print('=' * 52)
print()
print('OBSERVACAO: cada token exigiu recalculo completo')
print('de Q, K, V para todos os tokens anteriores.')

---
## Passo 4 — A Solucao: KV Cache + SDPA (Scaled Dot Product Attention)

**KV Cache (`use_cache=True`):** armazena os tensores K e V entre os passos de geracao.
Cada passo de decoding passa apenas o **novo token** pelo modelo — custo O(1) por passo.

**SDPA — `attn_implementation='sdpa'`:** implementacao nativa do PyTorch 2.x que despacha
automaticamente para o kernel de atencao mais eficiente disponivel na GPU, usando
o mesmo algoritmo de *tiling* em SRAM do FlashAttention-2. Complexidade de memoria: **O(n)**.

> **Nota:** `flash-attn` e o pacote externo que expoe explicitamente o FlashAttention-2.
> O `sdpa` do PyTorch e o equivalente embutido — mesma reducao de complexidade, sem compilacao separada.
> Para fins academicos deste laboratorio, os dois demonstram o mesmo principio arquitetural.


In [ ]:
print('Liberando modelo da VRAM...')
del model
del inputs_baseline
gc.collect()
torch.cuda.empty_cache()
print(f'VRAM apos limpeza: {torch.cuda.memory_allocated()/1024**2:.1f} MB')

# Forca o backend memory-efficient do SDPA (O(n) de memoria, sem compilacao externa)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

print()
print('Recarregando modelo COM SDPA memory-efficient...')
torch.cuda.reset_peak_memory_stats()

model_fa2 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    attn_implementation='sdpa',
)
model_fa2.eval()
model_fa2.config.use_cache = True

vram_modelo_fa2 = torch.cuda.memory_allocated() / 1024**2
print(f'Modelo SDPA carregado! VRAM: {vram_modelo_fa2:.1f} MB')

print()
print(f'Criando contexto de {TOKENS_FA2:,} tokens para Passo 4...')
inputs_fa2 = criar_input(TOKENS_FA2)


In [ ]:
print('[PASSO 4] Geracao COM KV Cache + FlashAttention-2')
print(f'Contexto: {TOKENS_FA2:,} tokens | Novos tokens: 100')
print()

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

t1 = time.perf_counter()

with torch.no_grad():
    out_com_cache = model_fa2.generate(
        **inputs_fa2,
        max_new_tokens=100,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id,
    )

t_com_cache = time.perf_counter() - t1
vram_peak_com = torch.cuda.max_memory_allocated() / 1024**2
n_novos_com = out_com_cache.shape[1] - inputs_fa2['input_ids'].shape[1]

print('[PASSO 4 - METRICA]')
print('=' * 52)
print(f'Contexto              : {TOKENS_FA2:,} tokens')
print(f'Novos tokens gerados  : {n_novos_com}')
print(f'Tempo total           : {t_com_cache:.2f} s')
print(f'Tokens por segundo    : {n_novos_com / t_com_cache:.2f}')
print(f'Pico de VRAM          : {vram_peak_com:.1f} MB')
print('=' * 52)

---
## Comparativo Final de Benchmark

Tabela consolidando as metricas dos Passos 3 e 4.

In [ ]:
sep = '=' * 72
lin = '-' * 72

print()
print(sep)
print('COMPARATIVO FINAL — LABORATORIO 10')
print(sep)
print(f'{"Metrica":<40} {"Sem Otimizacao":>15} {"Com Otimizacao":>15}')
print(f'{"":40} {"(Passo 3)":>15} {"(Passo 4)":>15}')
print(lin)
print(f'{"Contexto (tokens)":<40} {TOKENS_BASELINE:>15,} {TOKENS_FA2:>15,}')
print(f'{"KV Cache":<40} {"Nao":>15} {"Sim":>15}')
print(f'{"FlashAttention-2":<40} {"Nao":>15} {"Sim":>15}')
print(f'{"Tempo de geracao (s)":<40} {t_sem_cache:>15.2f} {t_com_cache:>15.2f}')
print(f'{"Tokens por segundo":<40} {n_novos_sem/t_sem_cache:>15.2f} {n_novos_com/t_com_cache:>15.2f}')
print(f'{"Pico de VRAM (MB)":<40} {vram_peak_sem:>15.1f} {vram_peak_com:>15.1f}')
print(lin)

speedup = t_sem_cache / t_com_cache
economia_vram = vram_peak_sem - vram_peak_com

print(f'Speedup          : {speedup:.1f}x mais rapido com KV Cache + FlashAttention-2')
print(f'Variacao de VRAM : {economia_vram:+.1f} MB no pico')
print(f'Escala de contexto: {TOKENS_FA2 // TOKENS_BASELINE}x maior com otimizacao ativa')
print(sep)

---
## Texto Gerado pelo Modelo

Saida produzida pelo modelo com KV Cache + FlashAttention-2 sobre o contexto medico de 12.000 tokens.

In [ ]:
texto_gerado = tokenizer.decode(
    out_com_cache[0][inputs_fa2['input_ids'].shape[1]:],
    skip_special_tokens=True,
)
print('Texto gerado (KV Cache + FA2):')
print('-' * 60)
print(texto_gerado)